# LIAR-PLUS — Merge splits & extract claims
Merges train / val / test TSV files into a single dataset, removes empty claims, and extracts the claim column for batch API runs.

In [1]:
# CELL 1 — Imports & column names
import pandas as pd

# Column names from LIAR-PLUS paper (Alhindi et al. 2018)
COLUMNS = [
    'id',
    'json_file',
    'label',
    'claim',
    'subject',
    'speaker',
    'job_title',
    'state',
    'party',
    'barely_true_count',
    'false_count',
    'half_true_count',
    'mostly_true_count',
    'pants_fire_count',
    'context',
    'justification'
]

print(f'Columns defined: {len(COLUMNS)}')

Columns defined: 16


In [2]:
# CELL 2 — Load splits, tag with split name, concatenate
splits = {
    'train': 'train2.tsv',
    'val':   'val2.tsv',
    'test':  'test2.tsv'
}

dfs = []
for split_name, filepath in splits.items():
    df = pd.read_csv(filepath, sep='\t', header=None, names=COLUMNS)
    df['split'] = split_name
    dfs.append(df)
    print(f'{split_name}: {len(df)} rows loaded')

merged = pd.concat(dfs, ignore_index=True)
print(f'\nTotal before cleaning: {len(merged)} rows')

train: 10242 rows loaded
val: 1284 rows loaded
test: 1267 rows loaded

Total before cleaning: 12793 rows


In [3]:
# CELL 3 — Remove rows with empty or NaN claims
before = len(merged)
merged['claim'] = merged['claim'].astype(str).str.strip()
merged = merged[merged['claim'].notna()]
merged = merged[merged['claim'] != '']
merged = merged[merged['claim'].str.lower() != 'nan']
merged = merged.reset_index(drop=True)

after = len(merged)
print(f'Rows removed (empty/NaN claim): {before - after}')
print(f'Rows remaining: {after}')
print(f'\nSplit distribution:')
print(merged['split'].value_counts())

Rows removed (empty/NaN claim): 2
Rows remaining: 12791

Split distribution:
split
train    10240
val       1284
test      1267
Name: count, dtype: int64


In [4]:
# CELL 4 — Add placeholder columns for API outputs
# Paraphrases (2 columns)
merged['openai_paraphrase']  = ''
merged['gemini_paraphrase']  = ''

# Justifications (4 columns)
merged['openai_just_on_original']    = ''
merged['openai_just_on_paraphrased'] = ''
merged['gemini_just_on_original']    = ''
merged['gemini_just_on_paraphrased'] = ''

print('Placeholder columns added.')
print(f'Total columns: {len(merged.columns)}')
print(merged.columns.tolist())

Placeholder columns added.
Total columns: 23
['id', 'json_file', 'label', 'claim', 'subject', 'speaker', 'job_title', 'state', 'party', 'barely_true_count', 'false_count', 'half_true_count', 'mostly_true_count', 'pants_fire_count', 'context', 'justification', 'split', 'openai_paraphrase', 'gemini_paraphrase', 'openai_just_on_original', 'openai_just_on_paraphrased', 'gemini_just_on_original', 'gemini_just_on_paraphrased']


In [5]:
# CELL 5 — Save full merged dataset (with all features + placeholders)
merged.to_excel('liar_plus_merged.xlsx', index=False)
print(f'Full dataset saved: liar_plus_merged.xlsx ({len(merged)} rows, {len(merged.columns)} columns)')

Full dataset saved: liar_plus_merged.xlsx (12791 rows, 23 columns)


In [6]:
# CELL 6 — Extract claim column only (for API batch input)
claims = merged[['id', 'claim']].copy()
claims.to_excel('claims_for_api.xlsx', index=False)
print(f'Claims file saved: claims_for_api.xlsx ({len(claims)} rows)')
print('\nFirst 3 claims:')
print(claims.head(3).to_string())

Claims file saved: claims_for_api.xlsx (12791 rows)

First 3 claims:
    id                                                                                                                                          claim
0  0.0                                                             Says the Annies List political group supports third-trimester abortions on demand.
1  1.0  When did the decline of coal start? It started when natural gas took off that started to begin in (President George W.) Bushs administration.
2  2.0                                      Hillary Clinton agrees with John McCain "by voting to give George Bush the benefit of the doubt on Iran."
